In [ ]:
import numpy as np
import scipy

K = 5
phi = np.pi / 4
angles = np.linspace(0, np.pi, K)

means = np.stack(
    [
        np.sin(phi) * np.cos(angles),
        np.sin(phi) * np.sin(angles),
        np.full(K, np.cos(phi)),
    ],
    axis=1,
)
sigmas = np.full(K, 0.3)
weights = np.full(K, 1 / K)

A = means[0]
B = means[-1]

In [2]:
def density(x, means, sigmas, weights):
    """x: (N, 2) -> p(x): (N,)"""
    p_array = np.array(
        [
            scipy.stats.multivariate_normal.pdf(
                x, mean=mean, cov=(sigma**2) * np.eye(x.shape[1])
            )
            for mean, sigma in zip(means, sigmas)
        ]
    )
    return np.sum(weights[:, np.newaxis] * p_array, axis=0)


def log_density(x, means, sigmas, weights):
    """log p(x). Returns shape (N,)."""
    p_array = np.array(
        [
            scipy.stats.multivariate_normal.logpdf(
                x, mean=mean, cov=(sigma**2) * np.eye(x.shape[1])
            )
            for mean, sigma in zip(means, sigmas)
        ]
    )
    return scipy.special.logsumexp((np.log(weights[:, np.newaxis]) + p_array), axis=0)


def score(x, means, sigmas, weights):
    """Gradient of log p at x. Returns shape (N, 2)."""
    p = density(x, means, sigmas, weights)
    derivative = np.array(
        [
            scipy.stats.multivariate_normal.pdf(
                x, mean=mean, cov=(sigma**2) * np.eye(x.shape[1])
            )[:, np.newaxis]
            * (mean - x)
            / (sigma**2)
            for mean, sigma in zip(means, sigmas)
        ]
    )
    return (
        np.sum(weights[:, np.newaxis, np.newaxis] * derivative, axis=0)
        / p[:, np.newaxis]
    )


def check_score(x, means, sigmas, weights, eps=1e-5):
    """Compare score() with a finite-difference gradient of log_density().
    Returns the largest absolute difference."""
    s = score(x, means, sigmas, weights)
    grad = np.zeros_like(s)
    for i in range(x.shape[1]):
        x_plus = x.copy()
        x_plus[:, i] += eps
        x_minus = x.copy()
        x_minus[:, i] -= eps
        grad[:, i] = (
            log_density(x_plus, means, sigmas, weights)
            - log_density(x_minus, means, sigmas, weights)
        ) / (2 * eps)
    return np.max(np.abs(s - grad))

In [3]:
x = np.random.rand(10, 3)
check_score(x, means, sigmas, weights)

np.float64(1.0435345920711825e-09)

In [ ]:
import plotly.graph_objects as go


def plot_sphere_landscape(
    means,
    sigmas,
    weights,
    paths=None,
    radius=1.0,
    n_grid=150,
    n_arrows=12,
    arrow_len=0.15,
    head_size=0.05,
):
    """paths: dict {label: array (M, 3)} or None. Returns a plotly Figure."""
    # opaque sphere coloured by density, without shading
    U, V = np.meshgrid(np.linspace(0, 2 * np.pi, n_grid), np.linspace(0, np.pi, n_grid))
    X, Y, Z = (
        radius * np.sin(V) * np.cos(U),
        radius * np.sin(V) * np.sin(U),
        radius * np.cos(V),
    )
    pts = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    P = density(pts, means, sigmas, weights).reshape(X.shape)
    fig = go.Figure(
        go.Surface(
            x=X,
            y=Y,
            z=Z,
            surfacecolor=P,
            colorscale="Viridis",
            cmin=0,
            opacity=1.0,
            lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
            colorbar=dict(title="p(x)"),
        )
    )
    # score arrows: white shaft + white cone head, tangent direction only
    if n_arrows > 0:
        Ua, Va = np.meshgrid(
            np.linspace(0, 2 * np.pi, 2 * n_arrows, endpoint=False),
            np.linspace(0.2, np.pi - 0.2, n_arrows),
        )
        pa = radius * np.stack(
            [
                (np.sin(Va) * np.cos(Ua)).ravel(),
                (np.sin(Va) * np.sin(Ua)).ravel(),
                np.cos(Va).ravel(),
            ],
            axis=1,
        )
        S = score(pa, means, sigmas, weights)
        n = pa / np.linalg.norm(pa, axis=1, keepdims=True)
        S = S - np.sum(S * n, axis=1, keepdims=True) * n  # tangent part only
        S = S / (np.linalg.norm(S, axis=1, keepdims=True) + 1e-12)  # direction only
        side = np.cross(n, S)  # tangent, perpendicular to S
        start = 1.02 * pa  # lifted above the surface
        end = start + arrow_len * S
        h = 0.35 * arrow_len  # size of the arrow head
        left = end - h * S + 0.6 * h * side
        right = end - h * S - 0.6 * h * side
        xs, ys, zs = [], [], []
        for p0, p1, l, r in zip(start, end, left, right):
            for a, b in [(p0, p1), (p1, l), (p1, r)]:  # shaft, then both head lines
                xs += [a[0], b[0], None]
                ys += [a[1], b[1], None]
                zs += [a[2], b[2], None]
        fig.add_trace(
            go.Scatter3d(
                x=xs,
                y=ys,
                z=zs,
                mode="lines",
                line=dict(color="white", width=2),
                showlegend=False,
            )
        )
    # paths (these keep their labels in the legend)
    if paths:
        for label, path in paths.items():
            p = 1.01 * path
            fig.add_trace(
                go.Scatter3d(
                    x=p[:, 0],
                    y=p[:, 1],
                    z=p[:, 2],
                    mode="lines+markers",
                    line=dict(width=7),
                    marker=dict(size=3),
                    name=label,
                )
            )
    fig.update_layout(scene=dict(aspectmode="data"), margin=dict(l=0, r=0, t=30, b=0))
    return fig

In [5]:
plot_sphere_landscape(means, sigmas, weights)

In [ ]:
def straight_path(A, B, M):
    """A, B: (2,) -> path: (M+1, 2)"""
    return np.linspace(A, B, M + 1)


def slerp_path(A, B, M):
    """A, B: (3,) on the sphere -> path: (M+1, 3) along the great circle"""
    s = np.linspace(0, 1, M + 1)[:, np.newaxis]
    A = A / np.linalg.norm(A)
    B = B / np.linalg.norm(B)
    dot = A @ B
    np.clip(dot, -1, 1)
    theta = np.arccos(dot)
    n_s = np.sin(theta)
    return (
        np.sin((1 - s) * theta) / n_s * A[np.newaxis, :]
        + np.sin((s) * theta) / n_s * B[np.newaxis, :]
    )


def path_length(path, means, sigmas, weights):
    """path: (M+1, 2) - weighted length (a single number)"""
    segment_lengths = np.linalg.norm(np.diff(path, axis=0), axis=1)
    midpoints = 0.5 * (path[:-1] + path[1:])
    densities = density(midpoints, means, sigmas, weights)
    return np.sum(segment_lengths / densities)

In [45]:
M = 10
slerp = slerp_path(A, B, M)
linear = straight_path(A, B, M)
print("slerp shape:", slerp.shape)
print("slerp length:", path_length(slerp, means, sigmas, weights))
print("linear length:", path_length(linear, means, sigmas, weights))
plot_sphere_landscape(means, sigmas, weights, paths={"slerp ": slerp})

slerp shape: (11, 3)
slerp length: 7.32016728485537
linear length: 4.921912670354065


In [65]:
times = np.linspace(0, 1, M + 1)
points = slerp

cs = scipy.interpolate.CubicSpline(times, points, axis=0, bc_type="natural")
spline = cs(np.linspace(0, 1, 8))
plot_sphere_landscape(means, sigmas, weights, paths={"spline": spline, "slerp": slerp})